In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
import faiss
import numpy as np
from torch.optim import AdamW

In [6]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
class BiEncoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids = input_ids,
            attention_mask = attention_mask
        )
        cls_embedding = outputs.last_hidden_state[:, 0]
        return cls_embedding

In [8]:
def tokenize(batch, tokenizer):
    return tokenizer(
        batch["query"],
        batch["positive_passages"]["passage_text"][0],
        padding = "max_length",
        truncation = True,
        max_length = MAX_LENGTH
    )

def prepare_dataset():
    dataset = load_dataset("ms_marco", "v2.1", split="train")
    dataset = dataset.select(range(100000))

    return dataset

In [13]:
def train():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = BiEncoder(MODEL_NAME).to(DEVICE)

    dataset = prepare_dataset()

    def collate_fn(batch):
        queries = [x["query"] for x in batch]
        positive_passage_texts = []
        for x_item in batch:
            passage_texts = x_item["passages"]["passage_text"]
            is_selected_flags = x_item["passages"]["is_selected"]
            found_positive = False
            for i, is_selected in enumerate(is_selected_flags):
                if is_selected == 1:
                    positive_passage_texts.append(passage_texts[i])
                    found_positive = True
                    break
            if not found_positive:
                positive_passage_texts.append(passage_texts[0] if passage_texts else "")

        q_tokens = tokenizer(
            queries,
            padding = True,
            truncation = True,
            max_length = MAX_LENGTH,
            return_tensors = "pt"
        )

        p_tokens = tokenizer(
            positive_passage_texts, # Corrected to use extracted positive passages
            padding = True,
            truncation = True,
            max_length = MAX_LENGTH,
            return_tensors = "pt"
        )

        return q_tokens, p_tokens
    loader = DataLoader(dataset, batch_size = BATCH_SIZE, shuffle=True, collate_fn = collate_fn)

    optimizer = AdamW(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler()

    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0
        loop = tqdm(loader)

        for q_tokens, p_tokens in loop:
            optimizer.zero_grad()

            q_input_ids = q_tokens["input_ids"].to(DEVICE)
            q_mask = q_tokens["attention_mask"].to(DEVICE)

            p_input_ids = p_tokens["input_ids"].to(DEVICE)
            p_mask = p_tokens["attention_mask"].to(DEVICE)

            with torch.cuda.amp.autocast():
                q_emb = model(q_input_ids, q_mask)
                p_emb = model(p_input_ids, p_mask)

                scores = torch.matmul(q_emb, p_emb.T)
                labels = torch.arange(scores.size(0)).to(DEVICE)

                loss = F.cross_entropy(scores, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} Loss: {total_loss/len(loader)}")
    torch.save(model.state_dict(), "bi_encoder.pt")
    print("-----------Model Saved-----------")

In [16]:
def build_index():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = BiEncoder(MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load("bi_encoder.pt"))
    model.eval()

    dataset = load_dataset("ms_marco", "v2.1", split="train")
    dataset = dataset.select(range(200000))

    # Corrected extraction of passages and flattening the list of lists
    all_passage_texts = []
    for item in dataset:
        all_passage_texts.extend(item['passages']['passage_text'])
    passages = list(set(all_passage_texts)) # Get unique passages for indexing

    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(passages), 64)):
            batch = passages[i:i+64]

            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            ).to(DEVICE)

            emb = model(tokens["input_ids"], tokens["attention_mask"])
            embeddings.append(emb.cpu().numpy())

    embeddings = np.vstack(embeddings)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, "msmarco.index")
    print("Index built and saved.")

In [11]:
def search(query, top_k=5):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = BiEncoder(MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load("bi_encoder.pt"))
    model.eval()

    index = faiss.read_index("msmarco.index")

    tokens = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        query_emb = model(tokens["input_ids"], tokens["attention_mask"])

    query_emb = query_emb.cpu().numpy()
    scores, indices = index.search(query_emb, top_k)

    return scores, indices

In [ ]:
if __name__ == "__main__":
    #train()
    build_index()
    # print(search("What is artificial intelligence?"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
 68%|██████▊   | 20133/29668 [2:44:08<1:17:12,  2.06it/s]